# Trader Performance vs. Bitcoin Market Sentiment Analysis

## Executive Summary
This notebook explores the relationship between trader performance and market sentiment. It integrates historical trading data with the Bitcoin Fear & Greed Index to identify actionable patterns and build predictive models for trader profitability.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import sys
import os

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_processing import load_data, clean_fear_greed, clean_historical, merge_datasets, get_profiling_summary
from src.features import run_feature_engineering
from src.eda import run_all_eda
from src.analysis import run_hypothesis_tests, segment_traders
from src.modeling import prepare_modeling_data, train_and_evaluate_models, plot_feature_importance

## Phase 1 & 2: Data Understanding and Cleaning
Load and clean the datasets.

In [ ]:
fg_path = '../fear_greed_index.csv'
hist_path = '../historical_data.csv'

fg_df, hist_df = load_data(fg_path, hist_path)

print("Fear & Greed Profiling:", get_profiling_summary(fg_df, "Fear & Greed"))
print("Historical Data Profiling:", get_profiling_summary(hist_df, "Historical"))

fg_clean = clean_fear_greed(fg_df)
hist_clean = clean_historical(hist_df)

## Phase 3 & 4: Feature Engineering and Dataset Integration
Merge the datasets based on trade dates and create sentiment and trader-specific features.

In [ ]:
merged_df = merge_datasets(fg_clean, hist_clean)
print(f"Merged Dataset Shape: {merged_df.shape}")

final_df = run_feature_engineering(merged_df)
print(f"Final Dataset Shape (with features): {final_df.shape}")
final_df.head()

## Phase 5: Exploratory Data Analysis (EDA)
Generate visualizations to understand distributions and correlations.

In [ ]:
run_all_eda(final_df, output_dir='../reports/figures')
print("Visualizations have been saved to the reports/figures directory.")

## Phase 6: Statistical Analysis
Test hypotheses regarding sentiment and trading performance.

In [ ]:
stats_results = run_hypothesis_tests(final_df)
for test, res in stats_results.items():
    print(f"\n{test}:")
    for k, v in res.items():
        print(f"  {k}: {v}")

## Phase 7: Trader Segmentation
Cluster traders based on their behaviors.

In [ ]:
trader_segments = segment_traders(final_df, output_dir='../reports/figures')
trader_segments.groupby('Cluster').mean()

## Phase 9: Predictive Modeling
Build machine learning models to predict if a trade will be profitable.

In [ ]:
X, y = prepare_modeling_data(final_df)
model_results, fitted_models, feature_names = train_and_evaluate_models(X, y)

results_df = pd.DataFrame(model_results).T
print(results_df)

plot_feature_importance(fitted_models, feature_names, output_dir='../reports/figures')